In [1]:
# --- [CELL 0]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 1}
import numpy as np
import pandas as pd
from pathlib import Path
import os.path
import matplotlib.pyplot as plt
from IPython.display import Image, display
import matplotlib.cm as cm

import tensorflow as tf 

import os
import shutil
from tqdm import tqdm
from random import shuffle

import cv2
from glob import glob

from tensorflow.keras import backend as K
import random
import albumentations as A
from sklearn.model_selection import train_test_split, StratifiedKFold

from tensorflow.keras.layers import *
from tensorflow.keras.optimizers import *
from tensorflow.keras.models import *
from tensorflow.keras.preprocessing.image import *
from tensorflow.keras.callbacks import *
from tensorflow.keras.applications.efficientnet import *

/usr/local/lib/python3.10/dist-packages/albumentations/__init__.py:13: UserWarning: A new version of Albumentations is available: 2.0.8 (you have 1.4.15). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()


In [2]:
# --- [CELL 1]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 2}
image_dir = Path('data_small/test')

# Get filepaths and labels
filepaths = list(image_dir.glob(r'**/*.jpg'))
labels = list(map(lambda x: os.path.split(os.path.split(x)[0])[1], filepaths))

filepaths = pd.Series(filepaths, name='Filepath').astype(str)
labels = pd.Series(labels, name='Label')

# Concatenate filepaths and labels
image_df = pd.concat([filepaths, labels], axis=1)

In [3]:
# --- [CELL 2]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 3}
# Shuffle the DataFrame and reset index
image_df = image_df.sample(frac=1).reset_index(drop = True)

# Show the result
image_df.head(5)

,Filepath,Label
0,data_small/test/tanks/613.jpg,tanks
1,data_small/test/tanks/592.jpg,tanks
2,data_small/test/cars/830.jpg,cars
3,data_small/test/tanks/650.jpg,tanks
4,data_small/test/cars/826.jpg,cars


In [4]:
# --- [CELL 3]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 4}
# Separate in train and test data
train_df, test_df = train_test_split(image_df, train_size=0.9, shuffle=True, random_state=1)

In [5]:
# --- [CELL 4]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 5}
train_generator = tf.keras.preprocessing.image.ImageDataGenerator(
    preprocessing_function=tf.keras.applications.mobilenet_v2.preprocess_input,
    validation_split=0.2
)

test_generator = tf.keras.preprocessing.image.ImageDataGenerator(
    preprocessing_function=tf.keras.applications.mobilenet_v2.preprocess_input
)

In [6]:
# --- [CELL 5]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 6}
train_images = train_generator.flow_from_dataframe(
    dataframe=train_df,
    x_col='Filepath',
    y_col='Label',
    target_size=(224, 224),
    color_mode='rgb',
    class_mode='categorical',
    batch_size=32,
    shuffle=True,
    seed=42,
    subset='training'
)

val_images = train_generator.flow_from_dataframe(
    dataframe=train_df,
    x_col='Filepath',
    y_col='Label',
    target_size=(224, 224),
    color_mode='rgb',
    class_mode='categorical',
    batch_size=32,
    shuffle=True,
    seed=42,
    subset='validation'
)

test_images = test_generator.flow_from_dataframe(
    dataframe=test_df,
    x_col='Filepath',
    y_col='Label',
    target_size=(224, 224),
    color_mode='rgb',
    class_mode='categorical',
    batch_size=32,
    shuffle=False
)

Found 44 validated image filenames belonging to 2 classes.
Found 10 validated image filenames belonging to 2 classes.
Found 6 validated image filenames belonging to 2 classes.


In [7]:
# --- [CELL 6]: ---
# cell_state: edited
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 7}
# === BEFORE (original) ===
# def create_model(input_shape=(224, 224, 3)):
#     
#     inputs = Input(input_shape)
#     base_model = EfficientNetB1(input_shape=input_shape, include_top=False, classes=5)
#     
#     x = base_model(inputs)
#     
#     x = GlobalAveragePooling2D()(x)
# #     x = Dropout(0.1)(x)
#     
#     x = Dense(56, activation='relu')(x)
#     x = Dropout(0.1)(x)
#     
#     outputs = Dense(5, activation='sigmoid')(x)
#     
#     model = Model(inputs, outputs)
#     
#     return model

# === AFTER (edited) ===
def create_model(input_shape=(224, 224, 3), num_classes=2):

    inputs = Input(input_shape)
    base_model = EfficientNetB1(input_shape=input_shape, include_top=False)

    x = base_model(inputs)
    x = GlobalAveragePooling2D()(x)

    x = Dense(56, activation='relu')(x)
    x = Dropout(0.1)(x)

    outputs = Dense(num_classes, activation='softmax')(x)

    model = Model(inputs, outputs)

    return model

In [8]:
# --- [CELL 7]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 8}
K.clear_session()

model = create_model((224, 224, 3))
# model = load_model('models/checkpoint/EfficientNetB0.h5')

metrics = [
    'accuracy',
    'AUC'
]

27018416/27018416 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [9]:
# --- [CELL 8]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 9}
model.compile(optimizer=Adam(), loss='categorical_crossentropy', metrics=metrics)

In [10]:
# --- [CELL 9]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 10}
checkpoint_path = 'model_224.keras'

callbacks = [
    EarlyStopping(monitor='val_loss', mode='min', patience=15, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', mode='min', factor=0.1, patience=5, min_lr=0.000001, verbose=1),
    ModelCheckpoint(monitor='val_loss', mode='min', filepath=checkpoint_path, verbose=1, save_best_only=True, save_weights_only=False)
]

In [11]:
# --- [CELL 10]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 11}
history = model.fit(
    train_images,
    validation_data=val_images,
    epochs=3, #30,
    callbacks=callbacks
)

Epoch 1/3


/usr/local/lib/python3.10/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 990ms/step - AUC: 0.5359 - accuracy: 0.5185 - loss: 0.7781
Epoch 1: val_loss improved from inf to 0.70998, saving model to model_224.keras
2/2 ━━━━━━━━━━━━━━━━━━━━ 46s 4s/step - AUC: 0.5544 - accuracy: 0.5350 - loss: 0.7621 - val_AUC: 0.4000 - val_accuracy: 0.4000 - val_loss: 0.7100 - learning_rate: 0.0010
Epoch 2/3
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - AUC: 1.0000 - accuracy: 1.0000 - loss: 0.1306   
Epoch 2: val_loss did not improve from 0.70998
2/2 ━━━━━━━━━━━━━━━━━━━━ 4s 3s/step - AUC: 1.0000 - accuracy: 1.0000 - loss: 0.1199 - val_AUC: 0.4600 - val_accuracy: 0.4000 - val_loss: 0.7207 - learning_rate: 0.0010
Epoch 3/3
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - AUC: 1.0000 - accuracy: 1.0000 - loss: 0.0469   
Epoch 3: val_loss did not improve from 0.70998
2/2 ━━━━━━━━━━━━━━━━━━━━ 4s 3s/step - AUC: 1.0000 - accuracy: 1.0000 - loss: 0.0432 - val_AUC: 0.5400 - val_accuracy: 0.4000 - val_loss: 0.8988 - learning_rate: 0.0010


In [12]:
# Verify model output dimension matches dataset class cardinality.
assert hasattr(train_images, 'class_indices'), 'train_images must expose class_indices'
expected_num_classes = len(train_images.class_indices)
actual_output_units = int(model.output_shape[-1])
assert actual_output_units == expected_num_classes, (
    f"Output units ({actual_output_units}) must match class count ({expected_num_classes})"
)

# Extra structural check on final Dense layer for fail-before/pass-after linkage.
assert hasattr(model.layers[-1], 'units'), 'Final layer should expose units'
assert int(model.layers[-1].units) == expected_num_classes, (
    f"Final Dense units ({model.layers[-1].units}) must equal class count ({expected_num_classes})"
)